# H-optimus-0: tempo di estrazione per vetrino (Colab o Kaggle)

Misura quanto costa estrarre gli embedding H-optimus-0 prima di lanciare tutti gli 899 vetrini (CAMELYON16 + 17, 8,35 milioni di tile, 2,2 TB da scaricare). Lo stesso notebook gira su **Colab** e su **Kaggle**: la piattaforma si riconosce da sola.

Tre misure, poi una proiezione:

1. **Solo il modello**: tile/s sulla GPU con input sintetico, per dimensione del batch. Dice anche quale batch usare.
2. **Pipeline completa** (`td task=stream`) su 7 vetrini scelti per coprire scanner, formato e dimensione. Tempi per fase: download, tiling + QC, embedding.
3. **Solo la lettura delle tile** (CPU) con 2, 4 e 8 thread, per capire se il collo di bottiglia è la GPU o la CPU.
4. **Proiezione** su tutti i vetrini da [`tile_counts.csv`](tile_counts.csv): ore di GPU, sessioni, spazio per gli embedding.

Gli embedding dei 7 vetrini sono veri: l'estrazione completa ([`extract_colab.ipynb`](extract_colab.ipynb)) li salta.

**Prima di iniziare**
- Accetta i termini su https://huggingface.co/bioptimus/H-optimus-0 e crea un token di lettura.
- Salva il token come secret `HF_TOKEN`: su Colab icona della chiave → *Notebook access* attivo; su Kaggle *Add-ons → Secrets*.
- Colab: *Runtime → Cambia tipo di runtime → GPU T4*. Kaggle: *Settings → Accelerator → GPU T4 x2* e *Internet on*.
- Solo dati pubblici: nessuna immagine clinica su Colab, Kaggle o un Drive personale.

In [ ]:
# 1. Piattaforma, percorsi e hardware
import os, sys, platform, shutil, subprocess
from pathlib import Path

KAGGLE = Path("/kaggle").exists()
PLATFORM = "kaggle" if KAGGLE else "colab"
if KAGGLE:
    CODE = Path("/kaggle/working/TumorDetector")
    OUT = Path("/kaggle/working/tumordetector")   # salvato come output del notebook (limite 20 GB)
    RAW = Path("/tmp/raw")                         # disco temporaneo, non salvato
else:
    from google.colab import drive
    drive.mount("/content/drive")
    CODE = Path("/content/TumorDetector")
    OUT = Path("/content/drive/MyDrive/tumordetector")   # sopravvive alle disconnessioni
    RAW = Path("/content/raw")
OUT.mkdir(parents=True, exist_ok=True)
OUTS = {"camelyon16": OUT, "camelyon17": OUT / "camelyon17"}   # stessa disposizione di data/ sul Mac

print(PLATFORM, platform.python_version())
!nvidia-smi --query-gpu=name,memory.total --format=csv
print("vCPU:", os.cpu_count())
!free -g | head -2
!df -h {RAW.parent} {OUT} | uniq

In [ ]:
# 2. Token Hugging Face dal secret HF_TOKEN (mai stampato né scritto su disco)
try:
    if KAGGLE:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    else:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as e:
    print("secret HF_TOKEN non disponibile:", type(e).__name__)
    from huggingface_hub import notebook_login
    notebook_login()

In [ ]:
# 3. Codice (se il repository è privato, clonalo con un token o rendilo pubblico)
REPO_URL = "https://github.com/masnarivan/TumorDetector.git"
REV = "main"    # meglio un commit preciso: finisce nei metadati di ogni vetrino
!git clone -q {REPO_URL} {CODE} 2>/dev/null || git -C {CODE} fetch -q
!git -C {CODE} checkout -q {REV}
# Il progetto chiede Python 3.12; l'immagine di Kaggle può essere più vecchia
flag = "" if sys.version_info >= (3, 12) else "--ignore-requires-python"
!pip install -q {flag} -e {CODE}
%cd {CODE}
COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("commit", COMMIT)

In [ ]:
# 4. Pesi (circa 4,5 GB): il tempo di download conta una volta per sessione
import time
from huggingface_hub import snapshot_download
t0 = time.time()
snapshot_download("bioptimus/H-optimus-0")
T_WEIGHTS = time.time() - t0
print(f"pesi pronti in {T_WEIGHTS:.0f} s")

## Misura 1 — solo il modello

Batch sintetici già normalizzati, float16 come nell'estrazione (`half: true`). Il tempo include il passaggio CPU → GPU → CPU, come in `encode`. Il batch più veloce che entra in memoria diventa `BATCH` per le misure successive.

In [ ]:
import pandas as pd
import torch
from omegaconf import OmegaConf
from tumordetector.backbones.models import build_backbone

bcfg = OmegaConf.load(CODE / "conf/backbone/h_optimus_0.yaml")
bb = build_backbone(bcfg)
t0 = time.time()
bb.load(torch.device("cuda"))
T_LOAD = time.time() - t0
print(f"modello caricato in {T_LOAD:.0f} s, pesi {bb.version}")

rows = []
for bs in [32, 64, 128, 256]:
    x = torch.randn(bs, 3, 224, 224)
    try:
        torch.cuda.reset_peak_memory_stats()
        bb.encode(x)                                   # riscaldamento
        torch.cuda.synchronize()
        n = max(4, 4096 // bs)
        t0 = time.time()
        for _ in range(n):
            bb.encode(x)
        torch.cuda.synchronize()
        rows.append({"batch": bs, "tiles_s": n * bs / (time.time() - t0),
                     "peak_gb": torch.cuda.max_memory_allocated() / 1e9})
    except torch.cuda.OutOfMemoryError:
        rows.append({"batch": bs, "tiles_s": float("nan"), "peak_gb": float("nan")})
        torch.cuda.empty_cache()
        break
model_only = pd.DataFrame(rows)
BATCH = int(model_only.loc[model_only.tiles_s.idxmax(), "batch"])
GPU = torch.cuda.get_device_name(0)
print(GPU, "→ batch", BATCH)
display(model_only.round(1))

transform = bb.transform          # serve alla misura 3
del bb
torch.cuda.empty_cache()          # la pipeline gira in un altro processo e le serve tutta la GPU

## Misura 2 — pipeline completa su 7 vetrini

Per ogni coppia dataset × scanner il vetrino con il numero di tile più vicino alla mediana (5 vetrini), più il vetrino al 90° percentile delle tile e `normal_108`, già estratto sul Mac (1580 tile in 260 s su MPS). Le tile sono quelle della griglia locale: la griglia dipende dal tiling (224 px a 0,5 µm/px), non dal backbone.

I vetrini restano sul disco (`keep_slides=true`) per la misura 3, poi si cancellano.

In [ ]:
counts = pd.read_csv(CODE / "notebooks/tile_counts.csv")

def closest(g, q):
    target = g.n_tiles.quantile(q)
    return g.assign(d=(g.n_tiles - target).abs()).sort_values(["d", "slide_id"]).iloc[0]

picked = [closest(g, 0.5) for _, g in counts.groupby(["dataset", "scanner"])]
picked.append(closest(counts, 0.9))
picked.append(counts[counts.slide_id == "normal_108"].iloc[0])
sample = pd.DataFrame(picked).drop(columns="d", errors="ignore").drop_duplicates("slide_id")
display(sample[["slide_id", "dataset", "center", "scanner", "size_gb", "n_tiles"]])
print(f"{sample.size_gb.sum():.1f} GB da scaricare, {sample.n_tiles.sum():,} tile")

In [ ]:
t_stream = {}
for ds, g in sample.groupby("dataset"):
    ids = ",".join(g.slide_id)
    out = OUTS[ds]
    t0 = time.time()
    !td task=stream backbone=h_optimus_0 data={ds} data.raw_dir={RAW} paths.out={out} \
        embed.batch_size={BATCH} data.keep_slides=true \
        mlflow.tracking_uri=sqlite:///{OUT}/mlflow.db mlflow.experiment=hoptimus-timing "data.slides=[{ids}]"
    t_stream[ds] = time.time() - t0
print({k: round(v) for k, v in t_stream.items()}, "s (incluso il caricamento del modello)")

In [ ]:
log = pd.concat([pd.read_csv(OUTS[ds] / "embeddings/h_optimus_0/extract_log.csv").assign(dataset=ds)
                 for ds in sample.dataset.unique()])
log = log.merge(sample[["slide_id", "scanner", "size_gb"]], on="slide_id")
done = log[log.status == "embedded"].copy()
if (log.status == "cached").any():
    print("vetrini già estratti in una sessione precedente, esclusi dalla misura:",
          list(log.slide_id[log.status == "cached"]))
done["download_MB_s"] = done.size_gb * 1000 / done.t_download_s
done["embed_tiles_s"] = done.n_tiles / done.t_embed_s
done["wall_s"] = done.t_wait_s + done.t_tile_s + done.t_embed_s
display(done[["slide_id", "scanner", "size_gb", "n_tiles", "t_download_s", "t_wait_s", "t_tile_s",
              "t_embed_s", "wall_s", "download_MB_s", "embed_tiles_s"]].round(1))
print(log[log.status != "embedded"][["slide_id", "status", "error"]].to_string() if (log.status != "embedded").any() else "")

## Misura 3 — solo la lettura delle tile

Lettura da OpenSlide + statistiche di colore per tile, senza modello, con 2, 4 e 8 thread (`embed.threads`, oggi 4). Ogni conteggio di thread legge un blocco diverso di 2048 tile, così la cache del disco non favorisce le prove successive. Se la lettura è più lenta della misura 1, il limite è la CPU e una GPU più veloce non serve.

In [ ]:
from tumordetector.embed.extract import TileDataset, iter_batches

rows = []
for r in sample.itertuples():
    tif = RAW / f"{r.slide_id}.tif"
    tiles = pd.read_parquet(OUTS[r.dataset] / "tiles" / f"{r.slide_id}.parquet")
    if not tif.exists() or len(tiles) < 3 * 2048:
        continue
    for k, th in enumerate([2, 4, 8]):
        ds = TileDataset(tif, tiles.iloc[k * 2048:(k + 1) * 2048], transform)
        t0 = time.time()
        for _ in iter_batches(ds, BATCH, th):
            pass
        rows.append({"slide_id": r.slide_id, "scanner": r.scanner, "threads": th, "tiles_s": 2048 / (time.time() - t0)})
reading = pd.DataFrame(rows)
display(reading.pivot_table(index=["slide_id", "scanner"], columns="threads", values="tiles_s").round(0))
shutil.rmtree(RAW, ignore_errors=True)   # i vetrini non servono più

## Proiezione su tutti gli 899 vetrini

Per ogni vetrino: tempo = max(download, tiling + embedding), perché `stream` scarica il vetrino successivo mentre elabora quello corrente. Le velocità sono quelle misurate sui 7 vetrini (somma delle tile / somma dei secondi); il tiling è proporzionale alle tile solo in prima approssimazione. Lo spazio viene dai file scritti davvero.

In [ ]:
import datetime, json

embed_rate = done.n_tiles.sum() / done.t_embed_s.sum()          # tile/s, lettura + modello
tile_rate = done.n_tiles.sum() / done.t_tile_s.sum()            # tile/s, maschera + griglia + QC
dl_rate = done.size_gb.sum() / done.t_download_s.sum()          # GB/s

def disk_bytes(r):
    root = OUTS[r.dataset] / "embeddings/h_optimus_0"
    return sum((root / f"{r.slide_id}{ext}").stat().st_size for ext in (".npy", ".parquet", ".json"))
bytes_per_tile = sum(disk_bytes(r) for r in done.itertuples()) / done.n_tiles.sum()

proj = counts.assign(t_s=lambda d: pd.concat([d.size_gb / dl_rate,
                                               d.n_tiles / tile_rate + d.n_tiles / embed_rate], axis=1).max(axis=1),
                     gpu_bound=lambda d: d.n_tiles / tile_rate + d.n_tiles / embed_rate >= d.size_gb / dl_rate)
summary = proj.groupby("dataset").agg(slides=("slide_id", "size"), tiles=("n_tiles", "sum"), gb=("size_gb", "sum"),
                                      hours=("t_s", lambda s: s.sum() / 3600), gpu_bound=("gpu_bound", "mean"))
summary.loc["totale"] = [summary.slides.sum(), summary.tiles.sum(), summary.gb.sum(), summary.hours.sum(),
                         (proj.gpu_bound).mean()]
summary["embedding_GB"] = summary.tiles * bytes_per_tile / 1e9
summary["sessioni_12h"] = summary.hours / 12
if KAGGLE:
    summary["settimane_Kaggle_30h"] = summary.hours / 30
display(summary.round(2))
print(f"{GPU}: modello {model_only.tiles_s.max():.0f} tile/s, pipeline {embed_rate:.0f} tile/s, "
      f"tiling {tile_rate:.0f} tile/s, download {dl_rate * 1000:.0f} MB/s, {bytes_per_tile:.0f} byte/tile su disco")

stamp = datetime.date.today().isoformat()
res_dir = OUT / "timing"
res_dir.mkdir(exist_ok=True)
name = f"{stamp}_{PLATFORM}_{GPU.replace(' ', '-')}"
done.to_csv(res_dir / f"{name}_slides.csv", index=False)
(res_dir / f"{name}.json").write_text(json.dumps({
    "platform": PLATFORM, "gpu": GPU, "vcpu": os.cpu_count(), "commit": COMMIT, "batch": BATCH,
    "t_weights_s": round(T_WEIGHTS), "t_load_s": round(T_LOAD),
    "model_only": model_only.round(2).to_dict("records"), "reading": reading.round(1).to_dict("records"),
    "rates": {"embed_tiles_s": embed_rate, "tile_tiles_s": tile_rate, "download_GB_s": dl_rate,
              "bytes_per_tile": bytes_per_tile},
    "projection": summary.round(3).reset_index().to_dict("records")}, indent=1, default=float))
print("salvato in", res_dir / f"{name}.json")

## Come leggere i risultati

- **pipeline ≪ modello** e la lettura (misura 3) è vicina alla pipeline: il limite è la CPU. Più thread, o su Kaggle due processi (uno per GPU) solo se i vCPU bastano.
- **pipeline ≈ modello**: il limite è la GPU. Su Kaggle T4 x2 due processi con `CUDA_VISIBLE_DEVICES=0` e `=1` su metà dei vetrini ciascuno dimezzano il tempo.
- **`gpu_bound` basso**: il download domina, la GPU aspetta la rete.
- **`embedding_GB`** contro lo spazio disponibile: 15 GB del Drive gratuito, 20 GB di output su Kaggle.

Riporta sul Mac la cartella `tumordetector/timing` (piccola) per confrontare Colab e Kaggle.